# CNN-LSTM

## Preprocessing

In [16]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical

In [17]:
con = duckdb.connect(database='D:/Assets/BinanceFuturesData/binancefuturesdata.duckdb', read_only=True)

In [18]:
# BTCUSDT 2025년 1월~11월의 1분봉 데이터
data_query = "SELECT epoch_ms(timestamp) as time, open, high, low, close, volume FROM quote " \
"WHERE symbol = 'BTCUSDT' " \
"and epoch_ms(timestamp) >= '2025-01-01 00:00:00' " \
"and epoch_ms(timestamp) < '2025-12-01 00:00:00' " \
"ORDER BY timestamp;"
df = con.execute(data_query).fetchdf()

con.close()

In [19]:
# 원본 데이터 크기
df.shape

(480960, 6)

In [20]:
def add_indicators(df):
    df = df.copy()
    # 1분봉이므로 노이즈가 많아 짧은/긴 이동평균선 추가
    df['ma_7'] = df['close'].rolling(window=7).mean()
    df['ma_25'] = df['close'].rolling(window=25).mean()
    df['ma_99'] = df['close'].rolling(window=99).mean()
    
    # RSI 계산
    delta = df['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['rsi'] = 100 - (100 / (1 + rs))
    
    # 볼린저 밴드 (변동성 지표)
    df['std'] = df['close'].rolling(window=20).std()
    df['upper_band'] = df['ma_25'] + (df['std'] * 2)
    df['lower_band'] = df['ma_25'] - (df['std'] * 2)
    
    # 거래량 변화율
    df['vol_change'] = df['volume'].pct_change()

    df.dropna(inplace=True) # 지표 계산으로 생긴 NaN 제거
    return df

In [21]:
# 지표 추가
df = add_indicators(df)

In [23]:
df[-5:]

,time,open,high,low,close,volume,ma_7,ma_25,ma_99,rsi,std,upper_band,lower_band,vol_change
480955,2025-11-30 23:55:00,90353.4,90402.0,90347.8,90402.0,102.526,90464.585714,90547.212,90900.955556,43.149038,91.897665,90731.007331,90363.416669,-0.466610
480956,2025-11-30 23:56:00,90401.9,90432.3,90364.3,90364.3,117.141,90438.014286,90536.732,90891.594949,28.835300,99.904251,90736.540503,90336.923497,0.142549
480957,2025-11-30 23:57:00,90364.3,90406.0,90333.3,90372.2,128.229,90409.428571,90529.016,90882.506061,31.361052,106.091167,90741.198333,90316.833667,0.094655
480958,2025-11-30 23:58:00,90372.2,90372.2,90300.0,90316.9,184.376,90380.271429,90519.676,90873.522222,25.906040,114.519476,90748.714952,90290.637048,0.437865
480959,2025-11-30 23:59:00,90316.8,90342.7,90300.0,90320.6,141.079,90357.885714,90510.228,90864.514141,21.015918,122.052339,90754.332679,90266.123321,-0.234830


Signal classification

In [ ]:
# ---------------------------------------------------------
# [핵심] 분류 라벨링 (Labeling for Classification)
# 전략: 현재로부터 'N분 뒤' 가격이 'K%' 이상 오르면 매수(1), 내리면 매도(2), 아니면 관망(0)
# ---------------------------------------------------------
LOOK_AHEAD = 10      # 10분 뒤의 가격을 예측 목표로 설정
THRESHOLD = 0.002    # 0.2% 변동성 기준 (수수료 고려하여 설정 필요)

# 미래 수익률 계산 ((10분 뒤 가격 - 현재 가격) / 현재 가격)
df['future_ret'] = df['close'].shift(-LOOK_AHEAD) / df['close'] - 1

In [24]:
# 라벨 정의 함수
def get_class(ret):
    if ret > THRESHOLD:
        return 1  # Buy (Long)
    elif ret < -THRESHOLD:
        return 2  # Sell (Short)
    else:
        return 0  # Hold (No Action)

In [26]:
df['label'] = df['future_ret'].apply(get_class)

In [27]:
# 라벨링을 위해 사용한 future_ret 컬럼과 마지막 LOOK_AHEAD 만큼의 데이터(정답 없음) 제거
df.dropna(subset=['future_ret'], inplace=True)

In [28]:
# 클래스 분포 확인 (불균형 체크)
df['label'].value_counts()

label
0    394136
1     43515
2     43184
Name: count, dtype: int64

In [ ]:
# 무한대(inf)를 NaN으로 변환
df.replace([np.inf, -np.inf], np.nan, inplace=True)
# NaN이 포함된 행 제거
df.dropna(inplace=True)
# 남은 행 개수
len(df)

# 의문: 데이터는 연속된 값인데 중간에 빠져도 훈련 데이터로서 상관없는건지?

480834

Data normalization

In [ ]:
# ---------------------------------------------------------
# 데이터 정규화
# ---------------------------------------------------------
# 입력 피처 선택 (시간 제외, 미래 참조 데이터 제외)
feature_cols = ['open', 'high', 'low', 'close', 'volume', 'ma_7', 'ma_25', 'ma_99', 'rsi', 'upper_band', 'lower_band', 'vol_change']
target_col = 'label'

In [35]:
# Train/Test 분리 (시계열 순서 유지)
train_size = int(len(df) * 0.8)
train_df = df.iloc[:train_size]
test_df = df.iloc[train_size:]

In [36]:
# 스케일링 (StandardScaler 권장 - 이상치에 조금 더 강함)
scaler = StandardScaler()
train_X_scaled = scaler.fit_transform(train_df[feature_cols])
test_X_scaled = scaler.transform(test_df[feature_cols])

In [37]:
# 타겟은 스케일링 하지 않음 (0, 1, 2 정수이므로)
train_y_raw = train_df[target_col].values
test_y_raw = test_df[target_col].values

Sequence data conversion

In [38]:
# 시퀀스 생성 함수
def create_dataset(X, y, time_steps=60):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y[i + time_steps]) # 시퀀스 끝나는 시점의 라벨
    return np.array(Xs), np.array(ys)

In [39]:
TIME_STEPS = 60 # 과거 60분(1시간) 데이터를 보고 판단

X_train, y_train_int = create_dataset(train_X_scaled, train_y_raw, TIME_STEPS)
X_test, y_test_int = create_dataset(test_X_scaled, test_y_raw, TIME_STEPS)

One-Hot Encoding

In [40]:
# ---------------------------------------------------------
# One-Hot Encoding (분류 모델 필수)
# 0 -> [1, 0, 0], 1 -> [0, 1, 0], 2 -> [0, 0, 1]
# ---------------------------------------------------------
y_train = to_categorical(y_train_int, num_classes=3)
y_test = to_categorical(y_test_int, num_classes=3)

In [41]:
# 전처리 결과 출력
print(f"입력 데이터 형태 (X_train): {X_train.shape}  -> (샘플 수, {TIME_STEPS}분, {len(feature_cols)}개 특징)")
print(f"출력 데이터 형태 (y_train): {y_train.shape}  -> (샘플 수, 3개 클래스)")
print(f"테스트 데이터 형태 (X_test): {X_test.shape}")

입력 데이터 형태 (X_train): (384607, 60, 12)  -> (샘플 수, 60분, 12개 특징)
출력 데이터 형태 (y_train): (384607, 3)  -> (샘플 수, 3개 클래스)
테스트 데이터 형태 (X_test): (96107, 60, 12)


Class weight process

In [42]:
# 클래스 불균형이 심해서 클래스 가중치 계산해서 변수에 저장하는 단계가 필요
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# ---------------------------------------------------------
# 클래스 가중치 계산 (Class Weight Calculation)
# 목적: 데이터가 적은 Buy(1)/Sell(2) 클래스에 더 큰 중요도를 부여하여
#      모델이 "무조건 Hold(0)"만 찍는 것을 방지함.
# ---------------------------------------------------------

# 1) One-Hot Encoding된 y_train([0, 1, 0])을 다시 정수([1])로 변환
# 이유: compute_class_weight 함수는 1차원 정수 배열을 입력으로 받기 때문
y_train_int = np.argmax(y_train, axis=1)

# 2) 존재하는 클래스 종류 확인 (0: Hold, 1: Buy, 2: Sell)
classes = np.unique(y_train_int)

# 3) 가중치 계산 ('balanced' 모드 사용)
# 공식: n_samples / (n_classes * np.bincount(y))
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train_int
)

# 4) Keras 모델 학습 시 전달할 딕셔너리 형태로 변환 {0: 0.4..., 1: 3.7..., 2: 3.7...}
class_weights = dict(zip(classes, weights))

In [43]:
# 클래스 가중치 계산 결과
print(f"클래스(Key): {list(class_weights.keys())}")
print(f"가중치(Value): {list(class_weights.values())}")
print("-" * 40)
print(f"Hold(0) 가중치: {class_weights[0]:.4f} (데이터가 많으므로 낮음)")
print(f"Buy (1) 가중치: {class_weights[1]:.4f} (데이터가 적으므로 높음)")
print(f"Sell(2) 가중치: {class_weights[2]:.4f} (데이터가 적으므로 높음)")
print("-" * 40)

# 이제 이 'class_weights' 변수를 model.fit() 함수에 전달하면 됨

클래스(Key): [0, 1, 2]
가중치(Value): [0.4025810354915932, 3.826821089917714, 3.9259633542591743]
----------------------------------------
Hold(0) 가중치: 0.4026 (데이터가 많으므로 낮음)
Buy (1) 가중치: 3.8268 (데이터가 적으므로 높음)
Sell(2) 가중치: 3.9260 (데이터가 적으므로 높음)
----------------------------------------


## Model architecture

**가장 일반적인 직렬 아키텍처는 1D CNN 블록이 먼저 지역적 특징을 추출하고, 그 출력이 LSTM 블록으로 전달되어 시간적 종속성을 학습하는 형태**

- 입력 레이어: (시퀀스 길이 T, 특징 수 F) 형태의 입력.
- 1D CNN 블록 (특징 추출):
- Conv1D 레이어: 시퀀스를 따라 슬라이딩하며 단기적이고 지역적인 패턴 (예: 캔들 패턴, 갑작스러운 거래량 급증)을 추출합니다. 필터 수, 커널 크기, 활성화 함수(ReLU 권장)를 설정합니다.
- MaxPooling1D 또는 AveragePooling1D 레이어: 추출된 특징의 차원을 줄이고 가장 중요한 특징을 선택합니다.
- (선택적) 여러 Conv1D 레이어를 쌓아 계층적 특징을 추출할 수 있습니다.
- 데이터 형태 변환 (RepeatVector 또는 TimeDistributed):CNN의 출력을 LSTM의 입력에 맞게 시퀀스 형태로 변환합니다.
- 일반적으로 CNN의 출력을 RepeatVector를 통해 $T'$ 시간 단계만큼 반복하여 LSTM에 전달하거나, TimeDistributed 래퍼를 사용하여 각 시간 단계에 CNN을 적용할 수 있습니다.
- LSTM 블록 (시퀀스 모델링):
- LSTM 레이어: CNN에서 추출된 특징의 장기적인 시간적 종속성과 추세 변화를 학습합니다. 유닛 수와 return_sequences 설정(여러 LSTM을 쌓을 경우)을 지정합니다.
- 출력 레이어:
- 회귀: Dense 레이어 1개 (활성화 함수: linear).
- 분류: Dense 레이어 3개 (매수/매도/보류인 경우) (활성화 함수: Softmax).

## Training & Validation

- 손실 함수 (Loss Function):
- 회귀: MSE (Mean Squared Error) 또는 MAE (Mean Absolute Error).
- 분류: Categorical Cross-Entropy (원-핫 인코딩 시).
- 최적화 도구 (Optimizer): Adam, RMSprop 등을 사용합니다.
- 훈련: 훈련 데이터셋으로 모델을 훈련시키고, 검증 데이터셋으로 과적합을 모니터링합니다. Early Stopping을 적용하여 성능 향상이 없을 때 훈련을 중단합니다.
- 평가 지표:
- 회귀: RMSE, R-squared.
- 분류: 정확도(Accuracy), 정밀도(Precision), 재현율(Recall), F1-Score, 백테스팅 수익률.

## Backtesting



Signal generation
- 훈련된 모델은 실시간 데이터(또는 백테스팅 데이터)를 입력받아 예측 가격 또는 트레이딩 신호(매수/매도/보류)를 출력합니다.
- 신호 해석:
- 가격 예측 기반: 예측 가격이 현재 가격보다 일정 임계값 이상 높으면 '매수', 낮으면 '매도', 그 외는 '보류'로 해석하는 로직을 추가해야 합니다.
- 분류 기반: 모델이 예측한 신호를 직접 트레이딩 액션으로 사용합니다.

Backtesting
- 시뮬레이션 환경: 과거 데이터를 사용하여 모델의 트레이딩 전략을 시뮬레이션합니다.
- 핵심 지표: 총 수익률, 최대 낙폭(MDD, Maximum Drawdown), 샤프 비율(Sharpe Ratio), 승률 등을 분석하여 전략의 안정성과 수익성을 평가합니다.